# MiniMax H3 whole-frame image-to-video - birthday test set

**Author**: Konrad Jelen (kj) <br>
**Date**: 2026-10-25 <br>
**Model**: MiniMax H3, FL2VA pruned bf16 transformer (33B), Qwen3-VL-32B nvfp4 text encoder, video and audio VAEs, through ComfyUI's native `MiniMaxH3ImageToVideo` nodes <br>
**Photo set**: `henryk-urodziny-2026`, three working copies <br>

Renders one 10-second live-photo clip per birthday photo with a whole-frame image-to-video model: the model redraws the whole photo and animates it a little from a text prompt, instead of warping the face of a still as LivePortrait does. Every clip is measured and shown under its source photo. The notebook measures and does not rank; the Star Colonel judges whether a clip is believable and looks natural.

The graph conditions on the photo as its first frame, samples 243 frames at 24 fps on a 768 x 1024 canvas with 20 res_multistep steps, and decodes video and audio. The client then converts the native clip to 300 frames at 30 fps with FILM, so the delivered clip is 10 seconds at 30 fps. The prompt has two parts: the Star Colonel's generic sentence verbatim, then a part written for that photo from its motion plan, the way a real child would move in that moment.

## Approach

1. Pin the RTX PRO 5000 by UUID and start the model's ComfyUI server on it, so no render touches another card
2. Load the setup file that carries the launch command, port, graph and native rate, and write one two-part prompt per photo
3. Render each working copy through the graph, one clip per photo, with a fixed seed
4. Convert each native 24 fps clip to 30 fps with FILM
5. Measure each clip and show it under its photo for the Star Colonel to judge

## Outputs

- `data/processed/henryk-urodziny-2026/minimax-h3/<stem>.mp4` - one clip per working copy
- `data/processed/henryk-urodziny-2026/minimax-h3/<stem>.md` - sidecar with source, model, weights, prompt, negative prompt, seed, size, steps, cfg, GPU, time, peak VRAM and interpolation
- `data/processed/henryk-urodziny-2026/minimax-h3/checkpoints/<stem>.json` - checkpoint that restores a finished clip on a re-run
- `logs/image-to-video-render-minimax.log` - execution log of this notebook


## GPU Selection

Pins the RTX PRO 5000 72GB (nvidia-smi index 2, compute capability 12.0) by UUID before the project package imports torch. The ComfyUI server started from the setup file carries the same UUID, and this kernel makes no CUDA call of its own. The Configuration render below shows the device the server reports.


In [ ]:
import os

# Deterministic ordering so a UUID always names the same physical card
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# RTX PRO 5000 72GB, pinned by UUID from `nvidia-smi -L`; the setup file launches the server on the same UUID
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-1223ff00-ce71-874f-592e-a4429e9f47ba"


## Imports

All imports in one cell. The project package comes after the GPU pin because `henryk_media.animation.common` imports torch. The notebook calls only the module: the module talks to ComfyUI over HTTP and owns the graph, the server and the interpolation.


In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import os  # Relative paths for the photos and clips shown below
from pathlib import Path  # Setup file path

# Notebook display
from IPython.display import Image, Markdown, Video, display  # Photos and clips by path, never embedded

# Rich console output
from rich import box  # Table border style
from rich import print as rprint  # Formatted console printing
from rich.progress import (  # Progress bar components
    BarColumn,
    MofNCompleteColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeElapsedColumn,
)
from rich.table import Table  # Tabular output

# Project package
from henryk_media.animation import common  # Seed and output folder
from henryk_media.animation import image_to_video as i2v  # Setup, ComfyUI server, render
from henryk_media.config import INTERIM_DATA_DIR, PROJ_ROOT  # Project paths


## Reproducibility

Seeds Python, NumPy and PyTorch in this kernel by convention. The graph's only random input is the `RandomNoise` seed, which is fixed per clip and recorded in the sidecar and checkpoint, so a re-render on the same GPU and software reproduces the clip.


In [ ]:
common.set_seed(42)


## Configuration

Inputs, the graph's native size and length, the fixed seed, the prompts, and the ComfyUI server. The cell ends by starting the server, so its render can show the device the server reports.

- **Size** - 768 x 1024 is the model's native 3:4 portrait canvas: a 768-pixel narrow edge in multiples of 32 at the photo's aspect ratio
- **Length** - 243 frames is the native 17k+5 grid at 24 fps, 10.13 s, long enough for the delivered 300 frames at 30 fps
- **Seed** - one fixed value for all three clips, so a change in a clip comes from the prompt or the photo, not from noise


In [ ]:
# Inputs
PHOTO_SET = "henryk-urodziny-2026"  # folder under data/interim and data/processed
WORKING_COPIES = [  # PNG copies with EXIF orientation applied; originals stay in data/external
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8840.png",
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8564.png",
    INTERIM_DATA_DIR / PHOTO_SET / "IMG_8732.png",
]
SETUP_PATH = PROJ_ROOT / "tmp/workflows/image-to-video/setup-minimax.json"  # launch command, port, graph, native rate
SETUP = i2v.Setup.load(SETUP_PATH)

# Output
OUTPUT_DIR = common.output_dir(PHOTO_SET, "minimax-h3")  # <stem>.mp4, <stem>.md, checkpoints/

# Graph values
WIDTH = 768  # narrow edge of the native canvas, a multiple of 32
HEIGHT = 1024  # 3:4 portrait
LENGTH = 243  # 17k+5 frames at the native 24 fps = 10.13 s
SEED = 42  # RandomNoise noise_seed, fixed for every clip
STEPS = 20  # BasicScheduler steps from the official graph

# Prompts: the Star Colonel's generic sentence verbatim, then the part written for each photo
GENERIC = "take the photo and make a small 5-10s animated clip out of this, essentially making the image live;"
SPECIFIC = {
    "IMG_8564": "the boy in the centre holding the picture book keeps his proud birthday grin while his head sways and tilts by small amounts as he shifts his weight; at about one and a half seconds his eyes glance briefly at someone beside the photographer and his head follows partway, then his eyes return to the lens with one blink while his head follows back more slowly; the girl on the left keeps looking down at the book and the girl on the right keeps looking at the camera, and the other children around them make only small natural movements",
    "IMG_8732": "the boy sits at the table over his drawing and stays absorbed in it: his eyes move from the raised pencil in his right hand to the drawing and along it, his head follows the glances with small turns and a slightly lowered chin, and he blinks once; he does not smile",
    "IMG_8840": "the boy holding the jar keeps showing it to the camera: his head turns and dips a little toward the jar, his eyes drop to it and come back to the lens with one blink, and his grin softens a little while he looks at the jar and comes back when he looks into the camera; the girl on the right - let her not react to the thing held in the hands by the boy, she just looks for a short while and turns to another thing",
}
PROMPTS = {stem: f"{GENERIC} {specific}" for stem, specific in SPECIFIC.items()}

# No official negative prompt exists for this checkpoint: it is CFG-distilled and the graph has no
# negative conditioning input, so the text is recorded in the sidecar and was not applied.
NEGATIVE_PROMPT = "camera movement, zoom, changing face, morphing, distorted hands, extra fingers, text"

WEIGHTS = [
    "minimax_h3_fl2va_pruned_bf16.safetensors",
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "minimax_h3_video_vae_fp16.safetensors",
    "minimax_h3_audio_vae_fp32.safetensors",
]
SETTINGS = {  # recorded in every sidecar and checkpoint
    "steps": STEPS,
    "sampler": "res_multistep",
    "scheduler": "simple",
    "shift": "12 video, 3 audio (MiniMaxH3 model settings)",
    "cfg": "none: CFG-distilled checkpoint, BasicGuider with positive conditioning only",
    "negative_prompt": NEGATIVE_PROMPT,
    "negative_prompt_applied": "no: the graph has no negative conditioning input",
    "weights": ", ".join(WEIGHTS),
}

# ComfyUI server on the pinned GPU, listening on 127.0.0.1 only; it loads weights on the first render
server = i2v.Server(SETUP)
device, system = server.stats["devices"][0], server.stats["system"]

photo_lines = "\n".join(f"  [cadet_blue]{photo.name}[/cadet_blue]" for photo in WORKING_COPIES)
rprint(f'''[bold medium_purple]Configuration[/bold medium_purple]
[dim]{"─" * 40}[/dim]
[bold slate_blue1]Inputs[/bold slate_blue1]
  Working copies: [dim]{WORKING_COPIES[0].parent}[/dim]
{photo_lines}
  Setup: [dim]{SETUP_PATH}[/dim]

[bold slate_blue1]Graph[/bold slate_blue1]
  Model: [cyan]{SETUP.model}[/cyan]
  Revision: [grey70]{SETUP.revision}[/grey70]
  Size: [light_sea_green]{WIDTH} x {HEIGHT} px[/light_sea_green] [dim](native 3:4 canvas, narrow edge 768)[/dim]
  Length: [dark_sea_green]{LENGTH}[/dark_sea_green] frames at [light_sea_green]{SETUP.native_fps:g} fps[/light_sea_green] ([light_sea_green]{LENGTH / SETUP.native_fps:.2f} s[/light_sea_green])
  Sampler: [grey70]{SETTINGS["sampler"]}, {SETTINGS["scheduler"]}, {SETTINGS["steps"]} steps[/grey70]
  CFG and negative prompt: [dark_goldenrod]none[/dark_goldenrod] [dim](CFG-distilled, BasicGuider)[/dim]
  Seed: [dark_sea_green]{SEED}[/dark_sea_green]

[bold slate_blue1]Delivery[/bold slate_blue1]
  Output: [dark_sea_green]{i2v.FRAMES}[/dark_sea_green] frames at [light_sea_green]{i2v.FPS} fps[/light_sea_green] ([light_sea_green]{i2v.FRAMES / i2v.FPS:.2f} s[/light_sea_green])
  Interpolation: [grey70]{i2v.INTERPOLATION}[/grey70]
  Clips: [dark_sea_green]{len(WORKING_COPIES)}[/dark_sea_green] [dim](one per working copy)[/dim]

[bold slate_blue1]ComfyUI[/bold slate_blue1]
  Version: [grey70]{system["comfyui_version"]}[/grey70], PyTorch [grey70]{system["pytorch_version"]}[/grey70]
  Port: [grey70]{SETUP.port}[/grey70], log [dim]logs/image-to-video-server-minimax.log[/dim]

[bold]Device[/bold] [dim](reported by the ComfyUI server)[/dim]
  Using: [green]{device["type"]}:{device["index"]}[/green]
  GPU: [cyan]{device["name"]}[/cyan]
  Pinned UUID: [cyan]{SETUP.gpu}[/cyan]
  Memory: [yellow]{device["vram_total"] / 1e9:.1f} GB[/yellow]

[bold slate_blue1]Output[/bold slate_blue1]
  Clips: [dim]{OUTPUT_DIR}[/dim]
''')


## Data Loading

Reads each working copy for its size and embedded colour profile. `MiniMaxH3ImageToVideo` stretches the first frame to the 768 x 1024 canvas with cropping disabled, so the canvas keeps the whole photo; these copies are within 0.4 percent of 3:4, so the stretch is under 0.4 percent and the clip keeps the photo's framing.


In [ ]:
photo_table = Table(
    title="Working copies",
    caption="aspect is width / height; the graph canvas is 768 x 1024 (0.7500)",
    box=box.SIMPLE_HEAVY,
)
for column in ("Photo", "Size", "Aspect", "Colour profile"):
    photo_table.add_column(column, style="grey70" if column == "Photo" else None)
for photo in WORKING_COPIES:
    height, width = common.load_working_copy(photo).shape[:2]
    photo_table.add_row(
        photo.name,
        f"[light_sea_green]{width} x {height} px[/light_sea_green]",
        f"{width / height:.4f}",
        common.colour_profile(photo) or "none",
    )
rprint(photo_table)


## Prompts

One prompt per photo, in the Star Colonel's two-part structure: his generic sentence first, verbatim, then a part written for that photo from its motion-plan assessment in `tmp/workflows/advanced-liveportrait-build/motion-plans/<stem>.json`. The second part describes only how the child moves in that moment and how the other people behave; nothing else goes into the positive prompt.

The negative prompt is recorded for the sidecar but not applied: the checkpoint is CFG-distilled and this graph has one `BasicGuider` with the positive conditioning only, so it has no negative input. The model publishes no official negative prompt to prepend.


In [ ]:
prompt_lines = "\n\n".join(
    f"[bold slate_blue1]{photo.stem}[/bold slate_blue1]\n  [grey70]{PROMPTS[photo.stem]}[/grey70]"
    for photo in WORKING_COPIES
)
rprint(
    f"[bold medium_purple]Positive prompt per photo[/bold medium_purple]\n\n{prompt_lines}\n\n"
    f"[bold slate_blue1]Negative prompt[/bold slate_blue1] [dark_goldenrod](not applied: no negative conditioning in the graph)[/dark_goldenrod]\n"
    f"  [grey70]{NEGATIVE_PROMPT}[/grey70]"
)


## Model

The setup file points at the graph the module fills and submits: native ComfyUI MiniMax H3 image-to-video with the pruned bf16 transformer, the nvfp4 Qwen3-VL-32B text encoder and the video and audio VAEs. The sampler is `res_multistep` with the `simple` scheduler at 20 steps, and the shift values of 12 for video and 3 for audio come from the model class, so the graph does not set them. The checkpoint is CFG-distilled, so the graph has one guider with the positive conditioning only.


In [ ]:
setup_table = Table(title="MiniMax H3 graph and server", box=box.SIMPLE_HEAVY)
setup_table.add_column("Setting", style="grey70")
setup_table.add_column("Value")
for key, value in (
    ("Model", f"[cyan]{SETUP.model}[/cyan]"),
    ("Revision", SETUP.revision),
    ("GPU", SETUP.gpu),
    ("Server", f"127.0.0.1:{SETUP.port}"),
    ("Graph", f"[dim]{SETUP.graph}[/dim]"),
    ("Native rate", f"[light_sea_green]{SETUP.native_fps:g} fps[/light_sea_green]"),
    ("Size", f"[light_sea_green]{WIDTH} x {HEIGHT} px[/light_sea_green]"),
    ("Frames", f"[dark_sea_green]{LENGTH}[/dark_sea_green]"),
    ("Sampler", f"{SETTINGS['sampler']}, {SETTINGS['scheduler']}, {SETTINGS['steps']} steps"),
    ("Shift", SETTINGS["shift"]),
    ("CFG", SETTINGS["cfg"]),
    ("Weights", ", ".join(f"[cadet_blue]{name}[/cadet_blue]" for name in WEIGHTS)),
):
    setup_table.add_row(key, value)
rprint(setup_table)


## Execution

Renders one clip per working copy: 3 renders of about 8 minutes each at 20 steps. For each clip the module copies the photo into the server's input folder, fills the graph's markers, submits it, polls until the history entry is finished, checks the native rate and duration, moves the native clip into the server's temp folder, converts it to 30 fps with FILM, writes the sidecar and checkpoint, and moves every file the render added or changed in the server's input, output and temp folders to the trash. Time runs from submission to the finished history entry; peak VRAM is sampled with nvidia-smi during the render. A clip whose checkpoint records the same model, revision and settings is restored instead of rendered again.

The 30 fps worker re-encodes video only, so the delivered clip has no sound track even though the graph generates one. The output folder held no earlier clip when this notebook was written; the module moves an earlier clip and sidecar of the same photo to the trash before writing a new one, so each photo entry is the latest render.


In [ ]:
jobs = [(photo, PROMPTS[photo.stem]) for photo in WORKING_COPIES]
results, restored = [], set()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with Progress(
    SpinnerColumn(),
    TextColumn("[steel_blue]{task.description}"),
    BarColumn(bar_width=40),
    MofNCompleteColumn(),
    TimeElapsedColumn(),
    refresh_per_second=10,
) as progress:
    task = progress.add_task("Rendering clips", total=len(jobs))
    for photo, prompt in jobs:
        clip = OUTPUT_DIR / f"{photo.stem}.mp4"
        progress.update(task, description=f"Rendering {clip.name}")
        before = clip.stat().st_mtime_ns if clip.exists() else None
        result = server.render(
            photo,
            clip,
            "single",
            {
                "{{PROMPT}}": prompt,
                "{{WIDTH}}": WIDTH,
                "{{HEIGHT}}": HEIGHT,
                "{{LENGTH}}": LENGTH,
                "{{SEED}}": SEED,
            },
            SETTINGS,
        )
        # A restored clip is not rewritten, so its mtime is unchanged by the call
        if before is not None and clip.stat().st_mtime_ns == before:
            restored.add(result.clip)
        results.append(result)
        progress.advance(task)
    # force completion (prevents N-1 issue)
    progress.update(task, completed=len(jobs), description="Rendered clips")
    progress.refresh()


## Save/Export

Each render already wrote its clip, sidecar and checkpoint. This cell lists the output folder, lists every file left in ComfyUI's input, output and temp folders to show that no copy of a photo or clip remains there, and stops the ComfyUI server so the GPU is released.


In [ ]:
written = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
comfyui_files = sorted(server.work_files())
server.stop()
file_lines = "\n".join(
    f"  [cadet_blue]{path.relative_to(OUTPUT_DIR)}[/cadet_blue] [dim]{path.stat().st_size / 1e6:.2f} MB[/dim]"
    for path in written
)
comfyui_lines = "\n".join(f"  [cadet_blue]{path.relative_to(SETUP.io_dirs["input"].parent)}[/cadet_blue]" for path in comfyui_files)
stopped = not server.answers()
rprint(
    f"[bold medium_purple]Output folder[/bold medium_purple] [dim]{OUTPUT_DIR}[/dim]\n{file_lines}\n\n"
    f"[bold slate_blue1]Files in ComfyUI's input, output and temp folders[/bold slate_blue1] [dim](ComfyUI's own files only)[/dim]\n"
    f"{comfyui_lines or '  [dim]none[/dim]'}\n\n"
    f"[bold slate_blue1]ComfyUI server[/bold slate_blue1]\n"
    f"  Stopped: [{'dark_sea_green' if stopped else 'indian_red'}]{'yes' if stopped else 'no'}[/]"
)


## Results

Measurements per clip, then each clip under its source photo for the Star Colonel to play.

- **Seconds** - wall seconds from submitting the graph to the finished history entry: text encoding, sampling of 243 frames at 768 x 1024 with 20 steps, VAE decode of video and audio, and the H.264 encode; model loading is included in the first clip
- **Peak VRAM** - the highest nvidia-smi `memory.used` of the pinned GPU sampled during the render, minus the memory in use before the server started
- **Interpolation** - FILM fp16 maps each 30 fps output frame to its exact position between native frames; a frame that lands on a native frame is the native frame itself


In [ ]:
result_table = Table(title="MiniMax H3 clips", box=box.SIMPLE_HEAVY)
for column in ("Clip", "Seconds", "Peak VRAM", "Size", "Frames", "FPS", "Duration", "Native rate", "Measured"):
    result_table.add_column(column, justify="left" if column in ("Clip", "Measured") else "right")
for result in results:
    result_table.add_row(
        f"[grey70]{Path(result.clip).name}[/grey70]",
        f"[light_sea_green]{result.seconds:.1f} s[/light_sea_green]",
        f"[light_sea_green]{result.peak_vram_gb:.2f} GB[/light_sea_green]",
        f"{result.width} x {result.height}",
        f"[dark_sea_green]{result.frames}[/dark_sea_green]",
        f"[dark_sea_green]{result.fps:g}[/dark_sea_green]",
        f"[light_sea_green]{result.frames / result.fps:.2f} s[/light_sea_green]",
        f"[light_sea_green]{result.settings['native_fps']:g} fps[/light_sea_green]",
        "[dim]checkpoint[/dim]" if result.clip in restored else "this run",
    )
rprint(result_table)


In [ ]:
# Photo and clip linked by path relative to the notebook: no image or video bytes are stored here
for photo in WORKING_COPIES:
    clip = OUTPUT_DIR / f"{photo.stem}.mp4"
    display(Markdown(f"### {photo.stem}"))
    display(Image(url=os.path.relpath(photo), width=400))
    display(Video(os.path.relpath(clip), embed=False, width=400, html_attributes="controls loop muted"))


## Conclusions

LENGTH 243 is the method the 2026-10-26 production run used: one native pass of 243 frames at 24 fps, interpolated with FILM to 300 frames at 30 fps, 10.00 s. The numbers below come from that run, recorded in `logs/h3-10s.log` and the sidecars of `data/processed/henryk-urodziny-2026/minimax-h3-10s/`; they are not this notebook's last execution, which predates the 10 s change.

- **Rendered** - ten clips, each 300 frames at 30 fps (10.00 s) at 1024 x 768, each with a sidecar
- **Time** - 1248.8 to 1391.0 s per clip from submitting the graph to the finished history entry; the first clip took 1266 s
- **Peak VRAM** - 64.4 to 69.5 GB above the memory in use before the server started, on the 72 GB RTX PRO 5000
- **Method** - a single native pass of 243 frames at 24 fps; FILM fp16 places each of the 300 output frames on the native timeline
- **Judgement** - how believable a clip looks is not measured here; the Star Colonel judges the clips
